In [2]:
import pandas as pd

### Data cleaning motivations

#### Missing values in buyer_regions and seller_regions
- Since there are three types of missing data:
    1. "Uknown"
    2. "Unknown"
    3. nan

   we normalise into just "Unknown"

#### Total apt vs bwt
- Since there are ~200 records where sum_apt > sum_bwt, we create a new column: is_valid_pdt,
    where pdt stands for partner delivery time, and is measured by bwt - apt

> **NOTE**:
> On paper, since bwt for an individual parcel needs to be greater or equal to the apt, the sum
> totals of bwt should also be greater or equal to the apt, if number of parcels counted inside bwt
> and apt are equal. Presumably, this is true since each row is tied to a specific number of parcels,
> but in the off chance this assumption doesn't hold (or there are some other variables like incorrect
> time logging on the logistical partner's side). I think I'd rather becareful and not drop the rows,
> although query implementations can be a bit more complicated.

#### Rows where parcel_qty are 0
- Since every row where parcel_qty is zero also has sum_apt and sum_bwt zero, for metric purposes
  these rows don't hold any information value. For confusion, we can drop

#### dt is represented as a string
- For less confusion, just standardise this into a datetime. Since I don't have information on whether
  the dates are normalised to utc (or some other timezone) or in their local timezones, I'll just take
  them as is

In [16]:
LLM_TEST_DATASET = "../data/raw/llm_test_dataset_20260206-172226.csv"

llm_test_df = pd.read_csv(LLM_TEST_DATASET, parse_dates=['dt'])

In [18]:
def normalise_region(x):
    if pd.isna(x):
        return "Unknown"
        
    s = str(x).strip()
    if s.lower() in {"unkown", "uknown", "unknown"}:
        return "Unkown"

    return s

In [19]:
llm_test_df["buyer_region"] = llm_test_df["buyer_region"].apply(normalise_region)

In [20]:
llm_test_df["seller_region"] = llm_test_df["seller_region"].apply(normalise_region)

In [21]:
llm_test_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 54258 entries, 0 to 54257
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   dt                  54258 non-null  datetime64[us]
 1   buyer_country       54258 non-null  str           
 2   buyer_region        54258 non-null  str           
 3   seller_country      54258 non-null  str           
 4   seller_region       54258 non-null  str           
 5   logistics_provider  54258 non-null  str           
 6   parcel_qty          54258 non-null  int64         
 7   sum_apt             54258 non-null  float64       
 8   sum_bwt             54258 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1), str(5)
memory usage: 3.7 MB


In [22]:
llm_test_df[(llm_test_df["buyer_region"] == "Uknown") | (llm_test_df["seller_region"] == "Uknown")].count()

dt                    0
buyer_country         0
buyer_region          0
seller_country        0
seller_region         0
logistics_provider    0
parcel_qty            0
sum_apt               0
sum_bwt               0
dtype: int64

In [26]:
# valid if >= 0 (sum_bwt >= sum_apt)
llm_test_df["is_valid_pdt"] = (llm_test_df["sum_bwt"] - llm_test_df["sum_apt"] > 0)

In [28]:
llm_test_df.head()

,dt,buyer_country,buyer_region,seller_country,seller_region,logistics_provider,parcel_qty,sum_apt,sum_bwt,is_valid_pdt
0,2025-01-01,ID,West Java,ID,Banten,J&T,31945,22125.70,46667.10,True
1,2025-01-01,MY,Johor,MY,Pahang,Ninja Van,22112,17236.96,37398.05,True
2,2025-01-01,MY,Selangor,MY,Negeri Sembilan,Pos Malaysia,16639,14421.88,25091.53,True
3,2025-01-01,TH,Songkhla,TH,Korat,Kerry Express,18926,12963.18,88948.68,True
4,2025-01-01,MY,Sarawak,MY,Penang,Ninja Van,21664,14648.78,111298.19,True


In [31]:
before = len(llm_test_df)
llm_test_df = llm_test_df[~(llm_test_df["parcel_qty"] == 0)]
after = len(llm_test_df)
print(f"Dropped {before - after} rows where parcel_qty == 0")

Dropped 171 rows where parcel_qty == 0


In [36]:
llm_test_df["dt"] = pd.to_datetime(llm_test_df["dt"], errors="raise")

In [37]:
OUT_PATH = "../data/intermediate/llm_test_dataset_20260206-172226_cleaned.csv"

llm_test_df.to_csv(OUT_PATH, index=False)
print(f"Cleaned data written to: {OUT_PATH}")

Cleaned data written to: ../data/intermediate/llm_test_dataset_20260206-172226_cleaned.csv


In [38]:
check_df = pd.read_csv(OUT_PATH)
check_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 54087 entries, 0 to 54086
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   dt                  54087 non-null  str    
 1   buyer_country       54087 non-null  str    
 2   buyer_region        54087 non-null  str    
 3   seller_country      54087 non-null  str    
 4   seller_region       54087 non-null  str    
 5   logistics_provider  54087 non-null  str    
 6   parcel_qty          54087 non-null  int64  
 7   sum_apt             54087 non-null  float64
 8   sum_bwt             54087 non-null  float64
 9   is_valid_pdt        54087 non-null  bool   
dtypes: bool(1), float64(2), int64(1), str(6)
memory usage: 3.8 MB
